<a href="https://colab.research.google.com/github/praveena-muvva/ai-agents/blob/main/langgraph_example2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain_openai
!pip install Langgraph

In [ ]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('openaiapikey')

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model = "gpt-4.1-nano"
)

In [ ]:
# With Reducer: this is needed when multiple updates combine into the same key
# Eg: if you need 2 different critiques 1 for tone and 1 for something else and don't want the last one to override
# Or when your chat messages need to be saved instead of overriding
# You tell LG "When you get a new value for this key, add to it instead of overriding"


# Langgraph usage example
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END

# Every langgraph needs a state where you need to define all the variables you use in the code
# Stategraph takes in this typed dictionary
class WorkflowState(TypedDict):
  product_name: str
  draft: str
  critiques: Annotated[list[str], operator.add] # can be replaced with our own reducer functions too
  # Or use a LG built in add_messages function which helps in adding chat messages
  final_copy: str

def generate_productname(state: WorkflowState) -> dict:
  """Step0: Come up with a unique product name"""
  response = llm.invoke(
      f"Come up with a suitable and unique product name for a fictional AI meeting assistant application"
      f"Length of the product name should be between 2 to 10 characters"
  )
  return {"product_name": response.content}

# Define nodes
# The general/most used signature of a node is taking in the state dict and returning a disctionary
def generate_node(state: WorkflowState) -> dict:
  """Step1: Write an initial product description"""
  response = llm.invoke(
      f"Write a compelling product description (2-3) paragraphs for a fictional"
      f"AI product called '{state['product_name']}'."
      f"It's an AI meeting assistant that joins meetings and takes notes"
      f"Identifies action items and sends meeting summaries."
  )
  return {"draft": response.content}

def tone_critique(state: WorkflowState) -> dict:
  response = llm.invoke(f"Critique this for tone only: \n {state['draft']}")
  return {"critiques": [response.content]} #Wrapping as a list

def clarity_critique(state: WorkflowState) -> dict:
  response = llm.invoke(f"Critique this for clarity only: \n {state['draft']}")
  return {"critiques": [response.content]} #Wrapping as a list

def refine_node(state: WorkflowState) -> dict:
  """step3: Rewrite the description incorporating the critique."""
  combined = "\n".join(state["critiques"])
  response = llm.invoke(
      f"Rewrite this product description, addressing every point in the critique below. "
      f"Keep what works, fix what doesn't.\n\n"
      f"Original draft:\n{state['draft']}\n\n"
      f"Critique:\n{combined}\n\n"
      f"Rewritten description:"
  )
  return {"final_copy": response.content}


# Build graph
graph = StateGraph(WorkflowState)
graph.add_node("generateName", generate_productname)
graph.add_node("generate", generate_node)
graph.add_node("tonecritique", tone_critique)
graph.add_node("claritycritique", clarity_critique)
graph.add_node("refine", refine_node)

graph.add_edge(START, "generateName")
graph.add_edge("generateName", "generate")
graph.add_edge("generate", "tonecritique")
graph.add_edge("generate", "claritycritique")
graph.add_edge("tonecritique", "refine")
graph.add_edge("claritycritique", "refine")
graph.add_edge("refine", END)

# Compile the graph
app = graph.compile()


In [ ]:
# Visualize the graph topology
from IPython.display import Image, display

display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
# Run graph
result = app.invoke({"product_name": "MeetingMaster"})

print("***** Draft ****")
print(result["draft"])
print("***** Critique ****")
print(result["critiques"])
print("**** Final Version ****")
print(result["final_copy"])

In [ ]:
# Run graph after generate_productname
result = app.invoke({})

print("***** Draft ****")
print(result["draft"])
print("***** Critique ****")
print(result["critiques"])
print("**** Final Version ****")
print(result["final_copy"])

In [ ]:
#Checkpoints
# This is how LG gives you graph memory and durability. So far, once you invoke, the state of the graph is gone.
# If you want to pause or fix something when something goes wrong at any step, pause and get approval, replay each step - all of these can be achieved by
# Saving a snapshot of the state after every node executes, tied to a "thread"
# thread is an ID representing one continous session of your graph
# Every time you invoke with that thread_id, LG loads from that state instead of starting over
# Simplest example is MemorySaver for in-memory but in production, they will be saved in a more permanent storage

from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

In [ ]:
#Running with thread id
config = {"configurable": {"thread_id": "product-1"}}

result = app.invoke(
    {"product_name": "", "draft": "", "critiques": [], "final_copy": ""},
    config=config
)

In [ ]:
for state in app.get_state_history(config):
  print(state.values, state.next)

In [ ]:
# If your state includes a messages field with add_messages reducer, each new invoke() calls the same thread id and adds to the previous messages and hence remembering the conversation